<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Handling Missing Data in Python - Solution

---

### About
In this notebook, we will explore various techniques for identifying, visualizing, and handling missing data in Python using `pandas` and other libraries. Missing data is a common challenge in real-world data sets, and understanding how to address it effectively is a crucial skill for data analysis.

### Learning Objectives
- Identify and visualize missing data
- Understand patterns of missingness
- Implement deletion methods
- Apply simple imputation techniques
- Evaluate the impact of different approaches

### Notebook Guide
1. Setup and Load Data
2. Identifying Missing Data
3. Deletion Methods
4. Simple Imputation Methods
5. Imputation Based on Other Variables
6. Handling Mixed Data Types
7. Best Practices and Conclusion
8. Now You Try
9. Conclusion

# 1. Setup and Load Data

In [ ]:
# !pip install missingno

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# For visualization of missing data
import missingno as msno

# For evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Set the random seed for reproducibility
np.random.seed(42)

# Display settings
pd.set_option('display.max_columns', None)
sns.set(style="whitegrid")

Let's load a data set with missing values to work with. We'll use a modified version of the Titanic data set.

In [ ]:
# Load the Titanic data set

# url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
# titanic_df = pd.read_csv(url)

titanic_df = pd.read_csv('../data/titanic-adj.csv')

# Display the first few rows
titanic_df.head()

# 2. Identifying Missing Data

The first step in handling missing data is to identify and understand the extent of missingness in your data set.

In [ ]:
# Check for missing values in each column
titanic_df.isnull().sum()

In [ ]:
# Calculate the percentage of missing values in each column
titanic_df.isnull().mean() * 100

### Visualizing Missing Data

Visualization can help identify patterns in missingness. The [`missingno`](https://github.com/ResidentMario/missingno) library is particularly useful for this purpose.

In [ ]:
# Create a matrix visualization of missing values
msno.matrix(titanic_df, figsize=(10, 6));

In [ ]:
# Create a bar chart showing the count of missing values by column
msno.bar(titanic_df, figsize=(8, 6));

In [ ]:
# Create a heatmap to show correlations between missingness
msno.heatmap(titanic_df, figsize=(8, 6));

### What patterns of missingness can you see in the Titanic data set? Are there any relationships between missing values in different columns?

There are several patterns of missingness:

1. The `Cabin` column has the highest percentage of missing values (78%), which suggests that cabin information was not recorded for most passengers.
2. `Age` is missing for about 20% of passengers, which is a substantial proportion.
3. `Embarked` has a very small percentage of missing values (0.2%), indicating just a few missing entries.

From the heatmap, there doesn't appear to be a strong correlation between missingness in different columns. This suggests that missingness in one column is not strongly related to missingness in other columns. For example, a passenger with a missing `Age` value is not necessarily more likely to have a missing `Cabin` value.

The lack of strong correlation between missingness patterns could suggest that the data might be **Missing Completely At Random (MCAR)** or **Missing At Random (MAR)**, but we would need _domain knowledge_ to confirm this.

---Back to the slides!---

# 3. Deletion Methods

The simplest approach to handling missing data is to remove observations or variables with missing values.

### 3.1 Listwise Deletion (Complete-case analysis)

In [ ]:
# Remove all rows with any missing values
df_complete = titanic_df.dropna()

# Check how many rows remain
print(f"Original data set size: {len(titanic_df)}")
print(f"Complete data set size: {len(df_complete)}")
print(f"Percentage of data retained: {len(df_complete) / len(titanic_df) * 100:.2f}%")

### 3.2 Column Deletion

In [ ]:
# Create a copy of the original data set
df_column_deletion = titanic_df.copy()

# Remove columns with more than 30% missing values
threshold = 0.3
df_column_deletion = df_column_deletion.loc[:, df_column_deletion.isnull().mean() < threshold]

# Print the remaining columns
print(f"Original columns: {list(titanic_df.columns)}")
print(f"Remaining columns: {list(df_column_deletion.columns)}")

### 3.3 Dropping specific columns

Sometimes, we decide to drop specific columns based on domain knowledge or analysis requirements.

In [ ]:
# Drop the 'Cabin' column
df_no_cabin = titanic_df.drop(columns=['Cabin'])
df_no_cabin.head()

### 3.4 Dropping specific rows (subset of columns)

In [ ]:
# Drop rows where 'Age' is missing, but keep rows with missing values in other columns
df_age_complete = titanic_df.dropna(subset=['Age'])

# Check how many rows remain
print(f"Original data set size: {len(titanic_df)}")
print(f"Data set with complete Age values: {len(df_age_complete)}")
print(f"Percentage of data retained: {len(df_age_complete) / len(titanic_df) * 100:.2f}%")

### What are the advantages and disadvantages of the deletion methods you've implemented above?

**Advantages of deletion methods:**

1. **Simplicity**: Deletion methods are straightforward to understand and implement.
2. **No imputation bias**: By removing missing values rather than imputing them, we avoid introducing potential biases from imputation.
3. **Valid for MCAR data**: If data is Missing Completely At Random, listwise deletion produces unbiased parameter estimates (though with reduced efficiency).
4. **Computational efficiency**: Deletion methods require less computational resources than some imputation techniques.

**Disadvantages of deletion methods:**

1. **Data loss**: As seen in our implementation, listwise deletion resulted in losing about 80% of the data set, which is a significant reduction in sample size.
2. **Statistical power reduction**: Smaller sample sizes lead to reduced statistical power, making it harder to detect significant effects.
3. **Potential bias**: If data is not MCAR, deletion methods can introduce bias in the analysis.
4. **Information loss**: Deleted observations might contain valuable information for other variables that are not missing.
5. **Complete-case analysis bias**: When certain groups or demographics are more likely to have missing data, their underrepresentation in the analysis could lead to skewed results.

**Specific to column deletion:**
- **Loss of potentially important variables**: In the Titanic data set, dropping the 'Cabin' column might eliminate information that could be useful for predicting survival (e.g., cabin location might correlate with survival rates).

**Specific to subset deletion:**
- **Inconsistent sample size**: Using different subsets of the data for different analyses can lead to inconsistencies in results.
- **Selective analysis**: Analyzing only passengers with age information might introduce selection bias if age missingness is related to other variables.

---Back to the slides!---

# 4. Simple Imputation Methods

Instead of deleting data, we can replace missing values with estimates.

### 4.1 Mean/Median/Mode Imputation

In [ ]:
# Create a copy of the original data set
df_mean_imputed = titanic_df.copy()

# Impute missing values in the 'Age' column with the mean
mean_age = df_mean_imputed['Age'].mean()
df_mean_imputed['Age'] = df_mean_imputed['Age'].fillna(mean_age)

# Check that there are no missing values in the 'Age' column
print(f"Missing values in Age column: {df_mean_imputed['Age'].isnull().sum()}")

In [ ]:
# Create a copy of the original data set
df_median_imputed = titanic_df.copy()

# Impute missing values in the 'Age' column with the median
median_age = df_median_imputed['Age'].median()
df_mean_imputed['Age'] = df_mean_imputed['Age'].fillna(mean_age)

# Check that there are no missing values in the 'Age' column
print(f"Missing values in Age column: {df_median_imputed['Age'].isnull().sum()}")

In [ ]:
# Create a copy of the original data set
df_mode_imputed = titanic_df.copy()

# Impute missing values in the 'Embarked' column with the mode
mode_embarked = df_mode_imputed['Embarked'].mode()[0]  # Get the first mode (in case of multiple modes)
df_mode_imputed['Embarked'] = df_mode_imputed['Embarked'].fillna(mode_embarked)

# Check that there are no missing values in the 'Embarked' column
print(f"Missing values in Embarked column: {df_mode_imputed['Embarked'].isnull().sum()}")

Let's compare the distribution of `Age` before and after imputation.

In [ ]:
# Compare the original and imputed age distributions
plt.figure(figsize=(12, 6))

# Original age distribution (excluding missing values)
plt.subplot(1, 3, 1)
sns.histplot(titanic_df['Age'].dropna(), kde=True)
plt.title('Original Age Distribution')
plt.xlabel('Age')

# Mean imputed age distribution
plt.subplot(1, 3, 2)
sns.histplot(df_mean_imputed['Age'], kde=True)
plt.axvline(mean_age, color='red', linestyle='--', label=f'Mean ({mean_age:.1f})')
plt.title('Mean Imputed Age Distribution')
plt.xlabel('Age')
plt.legend()

# Median imputed age distribution
plt.subplot(1, 3, 3)
sns.histplot(df_median_imputed['Age'], kde=True)
plt.axvline(median_age, color='red', linestyle='--', label=f'Median ({median_age:.1f})')
plt.title('Median Imputed Age Distribution')
plt.xlabel('Age')
plt.legend()

plt.tight_layout()
plt.show()

### 4.2 Constant Value Imputation

In [ ]:
# Create a copy of the original data set
df_constant = titanic_df.copy()

# Impute missing 'Cabin' values with 'Unknown'
df_constant['Cabin'] = df_constant['Cabin'].fillna('Unknown')

# Check the result
df_constant['Cabin'].value_counts()

### How do mean/median/mode imputation and constant imputation affect the distribution of the data? What are potential issues with these methods?

**Effects on data distribution:**

1. **Mean/Median Imputation**:
   - Creates an artificial spike in the distribution at the imputed value (mean or median)
   - Reduces variance as all missing values receive the same value
   - Distorts the natural shape of the distribution
   - Preserves the mean (in mean imputation) but alters other statistical properties

2. **Mode Imputation** (for categorical variables):
   - Increases the frequency of the already most common category
   - May artificially strengthen the majority class in classification problems
   - Can distort relationships between categorical variables

3. **Constant Imputation**:
   - Creates an artificial category or value that didn't exist in the original data
   - Introduces a distinct group of observations with the same imputed value
   - Can create meaningful separation if missingness itself is informative

**Potential issues with these methods:**

1. **Underestimation of variance and standard errors**:
   - Single imputation methods treat imputed values as known with certainty
   - Understates the uncertainty in the data, leading to overly confident statistical inferences

2. **Distortion of relationships between variables**:
   - These methods do not consider relationships between variables
   - Can weaken correlations and covariances between variables
   - May bias regression coefficients and other multivariate statistics

3. **Artificial data structure**:
   - Creates unrealistic clusters or patterns in the data
   - May lead to incorrect conclusions about data distribution

4. **No accounting for uncertainty in missing values**:
   - Single imputation methods provide only one possible value for each missing entry
   - Fails to reflect the uncertainty about what the true values might have been

5. **Potentially biased estimates if MAR or MNAR**:
   - If missingness is related to other variables or to the missing values themselves, simple imputation methods can introduce significant bias

# 5. Imputation Based on Other Variables

More sophisticated imputation methods leverage relationships between variables to estimate missing values.

### 5.1 Imputation by Group

In [ ]:
# Create a copy of the original data set
df_group_imputed = titanic_df.copy()

# Group passengers by class and gender, then impute age with the group mean
group_means = df_group_imputed.groupby(['Pclass', 'Sex'])['Age'].transform('mean')
df_group_imputed['Age'] = df_group_imputed['Age'].fillna(group_means)

# Check if there are any missing age values left
print(f"Missing Age values after group imputation: {df_group_imputed['Age'].isnull().sum()}")

Let's compare the age distribution after group imputation with the original data.

In [ ]:
# Compare the original and group-imputed age distributions
plt.figure(figsize=(10, 6))

# Plot histograms
sns.histplot(titanic_df['Age'].dropna(), kde=True, label='Original (non-missing)', alpha=0.5)
sns.histplot(df_group_imputed['Age'], kde=True, label='Group imputed', alpha=0.5)

plt.title('Age Distribution: Original vs Group Imputation')
plt.xlabel('Age')
plt.legend()
plt.tight_layout()
plt.show()

### 5.2 Adding Missing Indicators

Creating indicator variables for missingness can help preserve information about the pattern of missing data.

In [ ]:
# Create a copy of the original data set
df_with_indicators = titanic_df.copy()

# Add missing indicators for 'Age' and 'Cabin'
df_with_indicators['Age_missing'] = df_with_indicators['Age'].isnull().astype(int)
df_with_indicators['Cabin_missing'] = df_with_indicators['Cabin'].isnull().astype(int)

# Impute missing values
df_with_indicators['Age'] = df_with_indicators['Age'].fillna(df_with_indicators['Age'].mean())
df_with_indicators['Cabin'] = df_with_indicators['Cabin'].fillna('Unknown')

# Check the result
df_with_indicators[['Age', 'Age_missing', 'Cabin', 'Cabin_missing']].head(10)

---Back to the slides!---

# 6. Handling Mixed Data Types

Real-world data sets often contain a mix of numerical and categorical variables. Let's implement a complete missing data handling strategy for the Titanic data set.

In [ ]:
# Create a copy of the original data set
df_complete_strategy = titanic_df.copy()

# Examine the data types
df_complete_strategy.dtypes

In [ ]:
# Implement a complete missing data strategy
# 1. Drop columns with too many missing values
df_complete_strategy = df_complete_strategy.drop(columns=['Cabin'])

# 2. Handle numerical variables
numeric_columns = ['Age', 'Fare']
# Group passengers by class and gender for more accurate imputation
for col in numeric_columns:
    # Calculate group means
    group_means = df_complete_strategy.groupby(['Pclass', 'Sex'])[col].transform('mean')
    # Impute missing values with group means
    df_complete_strategy[col] = df_complete_strategy[col].fillna(group_means)
    # For any remaining missing values (rare groups), use overall mean
    df_complete_strategy[col] = df_complete_strategy[col].fillna(df_complete_strategy[col].mean())

# 3. Handle categorical variables
# Impute 'Embarked' with mode
df_complete_strategy['Embarked'] = df_complete_strategy['Embarked'].fillna(df_complete_strategy['Embarked'].mode()[0])

# 4. Check if there are any missing values left
print("Missing values after complete strategy:")
print(df_complete_strategy.isnull().sum())

# 7. Best Practices and Conclusion

Let's summarize best practices for handling missing data in Python:

### Best Practices for Handling Missing Data

1. **Understand the data**: Always start by exploring the extent and patterns of missingness.
2. **Consider the missingness mechanism**: Is the data MCAR, MAR, or MNAR? This influences your choice of handling method.
3. **Document your approach**: Keep track of how you handled missing data for transparency and reproducibility.
4. **Use appropriate methods for different variables**: Consider the type and importance of each variable.
5. **Create missing indicators**: For key variables, create indicator variables to preserve information about missingness.
6. **Compare different approaches**: Evaluate the impact of different methods on your analysis or model.
7. **Consider domain knowledge**: Incorporate subject matter expertise in your imputation strategy.
8. **Be cautious with deletion methods**: Only use deletion when data is MCAR or the amount of missingness is very small.
9. **Conduct sensitivity analysis**: Test how sensitive your conclusions are to different missing data handling approaches.

# 8. Now You Try

1. Load a data set of your choice (you can use any that we have seen so far, or one of your own) and identify missing values.
2. Visualize the pattern of missingness using `missingno`.
3. Implement at least three different imputation methods.
4. Evaluate and compare the methods based on their effect on your analysis results or model performance.
5. Develop a comprehensive strategy for handling missing data in the data set, taking into account the variable types and missingness patterns.

# 9. Conclusion

In this notebook, we've explored various techniques for handling missing data in Python using pandas and other libraries. We've learned how to identify, visualize, and address missing values using deletion and imputation methods. Remember that there's no one-size-fits-all approach to handling missing data, and the best strategy depends on the specific characteristics of your data set and the goals of your analysis.